# Ensemble Accuracy vs Rho
This notebook visualizes the relationship between the parameter $\rho$ and two accuracy metrics for ensembles: the mean component accuracy and the ensemble accuracy.

In [13]:
import mlflow
import polars as pl
import plotly.graph_objects as go

# Set MLflow tracking URI
mlflow.set_tracking_uri("http://localhost:5000")

# Retrieve ensemble runs
experiment = mlflow.get_experiment_by_name("contopo")
models_pd = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id], filter_string="tags.kind = 'ensemble'"
)
ensembles = pl.from_pandas(models_pd)
ensembles = ensembles.filter(pl.col("params.method") == "soft")
ensembles = ensembles.filter(pl.col('tags.num_components') == '10')

# Select relevant columns
plot_df = ensembles.select([
    'tags.rho',
        'metrics.comp_mean_acc',
        'metrics.ensemble_accuracy'
])
plot_df = plot_df.sort('tags.rho')

In [14]:
# Plotting
fig = go.Figure()
fig.add_trace(go.Scatter(x=plot_df['tags.rho'], y=plot_df['metrics.comp_mean_acc'],
        mode='lines+markers', name='Mean Component Accuracy'))
fig.add_trace(go.Scatter(x=plot_df['tags.rho'], y=plot_df['metrics.ensemble_accuracy'],
        mode='lines+markers', name='Ensemble Accuracy'))
fig.update_layout(
        title='Ensemble and Component Accuracy vs Rho',
        xaxis_title='Rho',
        yaxis_title='Accuracy',
        template='simple_white'
        )
fig.show()